# ETHUSDT Quantitative Research Platform — Notebook 03
## Milestone 3: Live Shadow / Paper Platform & Champion-Challenger Dashboard

This notebook operates the live shadow and historical market replay trading environment.

### Supported Modes:
- `REPLAY`: Accelerated historical event replay through the live state engine (e.g. 1 day in 30 seconds).
- `PARITY_AUDIT`: Verify 100% Backtest/Live mathematical and event parity.
- `LIVE_STREAM`: Real-time WebSocket streaming from Binance USDⓈ-M Futures (Paper mode).
- `PORTFOLIO_DASHBOARD`: Inspect Champion vs Challengers comparative performance and drift status.

In [ ]:
# =============================================================================
# 0. GOOGLE COLAB / LOCAL REPOSITORY SYNC & SETUP
# =============================================================================
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
REPO_URL = "https://github.com/umutergul74/daytrader.git"
REPO_DIR = Path("/content/daytrader")

if IN_COLAB:
    print("🚀 [Google Colab Detected] Initializing Daytrader Platform...")
    if not REPO_DIR.exists():
        print(f"Cloning latest repository from {REPO_URL}...")
        !git clone {REPO_URL} /content/daytrader
    else:
        print("Pulling latest updates from GitHub...")
        !cd /content/daytrader && git pull

    os.chdir(str(REPO_DIR))
    print("Installing dependencies...")
    !pip install -q polars pandas numpy scipy scikit-learn lightgbm xgboost catboost pydantic pydantic-settings typer rich matplotlib pyarrow requests websockets pytest optuna

    if str(REPO_DIR / "src") not in sys.path:
        sys.path.insert(0, str(REPO_DIR / "src"))
    print(f"✓ Environment ready! Working directory: {Path.cwd()}")
else:
    project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    os.chdir(str(project_root))
    if str(project_root / "src") not in sys.path:
        sys.path.insert(0, str(project_root / "src"))
    print(f"✓ [Local Mode] Working directory: {Path.cwd()}")

import polars as pl
from datetime import datetime, timezone
from quant_platform.config.settings import settings
from quant_platform.data.storage.canonical import CanonicalStorage
from quant_platform.live.state_engine import LiveStateEngine
from quant_platform.live.parity import ParityEngine
from quant_platform.live.replay import MarketReplayEngine
from quant_platform.live.paper_broker import PaperBroker
from quant_platform.live.champion_challenger import ChampionChallengerCoordinator
from quant_platform.strategies.advanced.liquidity_sweep_fvg import LiquiditySweepFVGStrategy
from quant_platform.strategies.baselines.breakout import BreakoutSanityStrategy
from quant_platform.strategies.baselines.ema_trend import EmaTrendStrategy

settings.ensure_directories()
print("✓ Quant Platform live shadow environment ready!")

In [ ]:
# =============================================================================
# 1. LIVE SHADOW CONFIGURATION & PARAMETERS
# =============================================================================
MODE = "REPLAY"  # Options: 'REPLAY', 'PARITY_AUDIT', 'LIVE_STREAM', 'PORTFOLIO_DASHBOARD'

SYMBOL = "ETHUSDT"
REPLAY_BARS = 500
REPLAY_SPEED = 50.0  # e.g., 50.0 = 50x live speed, 0.0 = instant

INITIAL_CAPITAL = 10000.0
RISK_PER_TRADE = 0.01  # 1% risk per trade
ENABLE_TELEGRAM = True
SESSION_DURATION_MINUTES = 60

In [ ]:
# =============================================================================
# 2. LOAD DATASET & INITIALIZE CHAMPION-CHALLENGER COORDINATOR (WITH AUTO-FETCH)
# =============================================================================
storage = CanonicalStorage()
df_1m = storage.read_range(symbol=SYMBOL)

if df_1m.is_empty():
    print(f"⚠️ No local canonical data found for {SYMBOL}. Auto-fetching 2024-01 from Binance archive...")
    from quant_platform.data.providers.binance_archive import BinancePublicArchiveProvider
    provider = BinancePublicArchiveProvider()
    df_month = provider.fetch_month(SYMBOL, "1m", 2024, 1)
    if df_month is not None and not df_month.is_empty():
        storage.write_month_partition(df_month, year=2024, month=1, symbol=SYMBOL)
        df_1m = storage.read_range(symbol=SYMBOL)

if df_1m.is_empty():
    raise RuntimeError(f"Could not load canonical data for {SYMBOL}.")

print(f"Loaded Canonical Dataset: {len(df_1m):,} bars available for {SYMBOL}")

# Initialize Champion and Challengers
champion = LiquiditySweepFVGStrategy(left_bars=3, right_bars=3)
challengers = {
    "Breakout": BreakoutSanityStrategy(lookback_period=20),
    "EmaTrend": EmaTrendStrategy(fast_period=10, slow_period=30),
}
coordinator = ChampionChallengerCoordinator(
    champion_strategy=champion,
    challengers=challengers,
    initial_capital_per_slot=INITIAL_CAPITAL,
    enable_telegram=ENABLE_TELEGRAM,
)
state_engine = LiveStateEngine(symbol=SYMBOL)

In [ ]:
# =============================================================================
# 3. EXECUTE SELECTED LIVE SHADOW MODE
# =============================================================================
if MODE == "PARITY_AUDIT":
    df_subset = df_1m.tail(REPLAY_BARS)
    rep = ParityEngine.verify_parity(df_subset, symbol=SYMBOL)
    print(rep.summary)

elif MODE == "REPLAY":
    df_subset = df_1m.tail(REPLAY_BARS)
    replay_engine = MarketReplayEngine(state_engine=state_engine, speed_multiplier=REPLAY_SPEED)
    
    def on_bar(snap):
        coordinator.evaluate_live_bar(state_engine)
        
    print(f"Executing Market Replay across {len(df_subset)} bars...")
    res = replay_engine.replay(df_subset, on_bar_callback=on_bar)
    
    snap = state_engine.get_latest_snapshot()
    dt_utc = datetime.fromtimestamp(snap['last_finalized_1m'] / 1000.0, tz=timezone.utc).strftime('%H:%M UTC')
    
    print("\n=======================================================")
    print("ETH QUANT LIVE SHADOW STATUS")
    print("=======================================================")
    print(f"Binance Feed:             CONNECTED (Market Replay)")
    print(f"Symbol:                   {snap['symbol']} PERPETUAL")
    print(f"Price:                    ${snap['live_price']:,.2f}")
    print(f"Last Finalized 1m:        {dt_utc}")
    print(f"Data Lag:                 {snap['data_lag_ms']} ms")
    print(f"Market Regime:            {snap['regime']}")
    print(f"1H Structure:             {snap['structure_1h']}")
    print(f"15M Structure:            {snap['structure_15m']}")
    print(f"Champion Strategy:        {champion.metadata.strategy_id}")
    print(f"Open Shadow Positions:    {len(coordinator.champion_slot.broker.portfolio.open_positions)}")
    print(f"Total Signals Generated:  {coordinator.signals_generated_count}")
    print(f"Telegram Alerts:          {'CONNECTED' if ENABLE_TELEGRAM else 'DISABLED'}")
    print("=======================================================\n")
    
    # Print Comparative Performance Table
    print("--- CHAMPION VS CHALLENGERS PERFORMANCE ---")
    comp_rows = coordinator.get_comparative_table()
    for r in comp_rows:
        print(f"[{r.role}] {r.strategy_name:30} | Trades: {r.total_trades:2} | PnL: ${r.net_pnl_usdt:+,.2f} ({r.net_return_pct:+.2f}%) | WinRate: {r.win_rate_pct:.1f}% | MaxDD: {r.max_drawdown_pct:.2f}%")

elif MODE == "LIVE_STREAM":
    from quant_platform.live.websocket_client import BinanceFuturesWebSocketClient
    import asyncio
    
    ws_client = BinanceFuturesWebSocketClient(symbol=SYMBOL, state_engine=state_engine, coordinator=coordinator)
    print(f"Connecting to live Binance USDⓈ-M Futures WebSocket for {SYMBOL}...")
    print("Press Interrupt/Stop to exit.")
    await ws_client.connect_and_listen(max_duration_seconds=SESSION_DURATION_MINUTES * 60)